[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/expquad_regression.ipynb)

GPy `ExpQuad` naming vs `gpc` (CIP-0024).

# ExpQuad regression with `gpc` (CIP-0024)

GPy's lengthscale-first squared exponential is `ExpQuad`. In `gpc`, `kernel="eq"` (aliases `ExpQuad` / `expquad`) builds `CEqKern` so paths are `ExpQuad.*` instead of `rbf.*`.

GPy (conceptual)::

```python
k = GPy.kern.ExpQuad(1) + GPy.kern.Bias(1) + GPy.kern.White(1)
m = GPy.models.GPRegression(X, Y, k)
m.ExpQuad.lengthscale = 0.2
m.optimize()
```

`gpc` (signal-only string; noise on likelihood)::

```python
m = gpc.GPRegression(X, Y, kernel="eq")
m.set_by_path("ExpQuad.lengthscale", 0.2)
```

Same \(K_of_r\) as RBF; storage remains `inverseWidth` under the hood.

```bash
pip install matplotlib ipywidgets
```


In [ ]:
import importlib
import re
import subprocess
import sys


def _gpc_ver_tuple(v: str):
    """Parse ``0.1.0a10`` / ``0.1.0`` into a comparable tuple (final > any alpha)."""
    m = re.fullmatch(r"(\d+)\.(\d+)\.(\d+)(?:a(\d+))?", str(v))
    if not m:
        return (0, 0, 0, 0)
    maj, mir, pat, alpha = m.groups()
    pre = int(alpha) if alpha is not None else 10**9
    return (int(maj), int(mir), int(pat), pre)


_MIN = "0.1.0a10"
_need = _gpc_ver_tuple(_MIN)


def _ensure_gpc():
    try:
        import gpc as _gpc
        if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) >= _need:
            return _gpc
    except ImportError:
        pass
    # Prefer TestPyPI for pre-release alphas; real PyPI for dependencies (CIP-002C).
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            "--no-cache-dir",
            "--index-url",
            "https://test.pypi.org/simple/",
            "--extra-index-url",
            "https://pypi.org/simple/",
            f"gpc>={_MIN}",
        ]
    )
    if "gpc" in sys.modules:
        importlib.reload(sys.modules["gpc"])
    import gpc as _gpc
    if _gpc_ver_tuple(getattr(_gpc, "__version__", "0")) < _need:
        raise RuntimeError(
            f"need gpc>={_MIN}, got {getattr(_gpc, '__version__', '?')}"
        )
    return _gpc


gpc = _ensure_gpc()


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc

try:
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])
    from matplotlib import pyplot as plt
    from gpc.plot import plot_1d


In [ ]:
import numpy as np
import gpc

rng = np.random.default_rng(0)
X = np.linspace(0.0, 1.0, 20)[:, None]
Y = np.sin(2 * np.pi * X) + 0.05 * rng.normal(size=X.shape)

m = gpc.GPRegression(X, Y, kernel="eq")
m.set_by_path("ExpQuad.lengthscale", 0.2)
m.set_by_path("ExpQuad.variance", 1.0)
print("paths:", m.param_paths())
print("objective before:", m.objective())
m.optimize(iters=50, messages=True)
print("objective after:", m.objective())
mu, var = m.predict(X)
print("predict shapes:", mu.shape, var.shape, "mean var", float(var.mean()))
print(m)


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

# Side-by-side: ExpQuad paths vs RBF paths (same covariance family).
m_rbf = gpc.GPRegression(X, Y, kernel="rbf")
m_rbf.set_by_path("rbf.lengthscale", 0.2)
m_rbf.optimize(iters=50)

Xtest = np.linspace(-0.1, 1.1, 200)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m, X, Y, Xtest=Xtest, ax=axes[0], label="ExpQuad")
axes[0].set_title("kernel='eq' → ExpQuad.*")
plot_1d(m_rbf, X, Y, Xtest=Xtest, ax=axes[1], label="RBF")
axes[1].set_title("kernel='rbf' → rbf.*")
plt.tight_layout()
plt.show()
